# Multimodale Routenoptimierung

Dieses Notebook berechnet die optimal gewichtete Route (Kosten, Zeit, CO2) in einem multimodalen Transportnetzwerk.

## Zielsetzung
- Minimierung einer gewichteten Zielfunktion aus Kosten, Dauer und Emissionen.
- Einhaltung von Budget- und Zeitrestriktionen.

## Mathematische Grundlagen

### Mengen und Zustände

| Symbol | Bedeutung |
|---|---|
| $H$ | Hubs |
| $M$ | Modi: road, rail, air, ship |
| $M_h \subseteq M$ | am Hub $h$ verfügbare Modi (`supported_modes`) |
| $T\subseteq\{0,\ldots,T_{\max}\}$ | ereignisbasierte Zeitpunkte in Minuten ($T_{\max}=2880$ im Beispiel) |
| $L^{\text{trans}}$ | Transportverbindungen mit Fahrplan (`arc_templates` mit `arc_type="transport"`) |
| $L^{\text{tr}}$ | Transferverbindungen mit Fahrplan (`arc_templates` mit `arc_type="transfer"`) |
| $w$ | Gewicht der Sendung in Tonnen |
| $c_m$ | Kostenfaktor des Modus $m$ in EUR pro Tonnenkilometer |
| $e_m$ | Emissionsfaktor des Modus $m$ in kg CO$_2$ pro Tonnenkilometer |

Ein Zeitknoten ist $n=(h,m,t)$ mit $h\in H$, $m\in M_h$ und $t\in T$.

### Kanten des zeitexpandierten Netzes

$$A=A_{\text{trans}}\cup A_{\text{wait}}\cup A_{\text{transfer}}.$$

| Kantenart | Bedeutung | Datenbasis |
|---|---|---|
| $A_{\text{trans}}$ | Fahrt zwischen Hubs | `arc_templates` |
| $A_{\text{wait}}$ | Warten in Hub und Modus | `supported_modes` |
| $A_{\text{transfer}}$ | Moduswechsel am Hub | `arc_templates` |

#### Transportkanten

Eine Transportverbindung $l\in L^{\text{trans}}$ wird als Tupel definiert:

$$l=(h_l^-,h_l^+,m_l,d_l,\Delta t_l,D_l^{\text{day}}).$$

| Bestandteil | Bedeutung | Bedingung |
|---|---|---|
| $h_l^-$ | Start-Hub | $h_l^-\in H$ |
| $h_l^+$ | Ziel-Hub | $h_l^+\in H$ |
| $m_l$ | Transportmodus | $m_l\in M$ |
| $d_l$ | Distanz in km | $d_l\geq 0$ |
| $\Delta t_l$ | Fahrzeit in Minuten | positive ganze Zahl (`duration_min`) |
| $D_l^{\text{day}}$ | tägliche Abfahrtsminuten | Minuten von 0 bis 1439 (`departure_minutes`) |

$\Delta t_l$ ist die direkt im Arc-Template angegebene Fahrzeit. Die Distanz $d_l$ bleibt zusätzlich erhalten, damit Kosten und Emissionen weiterhin distanzbasiert berechnet werden können.


$P(T_{\max})$ bezeichnet die im Planungshorizont betrachteten Tage; ein Tag hat 1440 Minuten:

$$P(T_{\max})=\left\{0,\ldots,\left\lceil\frac{T_{\max}}{1440}\right\rceil-1\right\}.$$

Für eine tägliche Abfahrtsminute $\theta\in D_l^{\text{day}}$ gilt:

$$D_l=\{\theta+1440p\mid \theta\in D_l^{\text{day}},\ p\in P(T_{\max})\}.$$

$$A_{\text{trans}}=
\left\{((h_l^-,m_l,t),(h_l^+,m_l,t+\Delta t_l))\ \middle|\ 
l\in L^{\text{trans}},\ m_l\in M_{h_l^-}\cap M_{h_l^+},\ t\in D_l,\ t+\Delta t_l\leq T_{\max}\right\}.$$


#### Wartekanten

Beim ereignisbasierten Netzwerk werden Wartekanten nicht für jede einzelne Minute erzeugt. Für jeden Zustand $(h,m)$ werden nur die tatsächlich relevanten Ereigniszeiten $E_{h,m}$ gesammelt, sortiert und paarweise verbunden:

$$A_{\text{wait}}=
\left\{((h,m,t_i),(h,m,t_{i+1}))\ \middle|\ h\in H,\ m\in M_h,\ t_i,t_{i+1}\in E_{h,m}\text{ sind aufeinanderfolgend}\right\}.$$

Die Dauer einer Wartekante ist damit $t_{i+1}-t_i$ Minuten.


#### Transferkanten

Eine Transferkante beschreibt das Umladen am selben Hub von einem bisherigen Modus $m_1$ auf einen neuen Modus $m_2$. Sie wird als eigenes Arc-Template modelliert, bei dem `from` und `to` denselben Hub enthalten, z. B. `BER_BER_road_rail_transfer` mit `from="BER"`, `to="BER"`, `from_mode="road"`, `to_mode="rail"`.

| Symbol | Bedeutung |
|---|---|
| $r\in L^{\text{tr}}$ | Transfer-Arc-Template |
| $h_r$ | Hub des Transfers (`from=to`) |
| $m_r^-$ | Modus vor dem Umladen (`from_mode`) |
| $m_r^+$ | Modus nach dem Umladen (`to_mode`) |
| $D_r^{\text{day}}$ | tägliche Startminuten (`departure_minutes`) |
| $\tau^{\text{tr}}_r$ | Dauer dieses konkreten Umladens in Minuten (`duration_min`) |

Da die Zeitknoten in Minuten gemessen werden, ist `duration_min` eine positive ganze Zahl.

Über mehrere Planungstage ergeben sich die zulässigen Startzeitpunkte als

$$D^{\text{tr}}_r=\{\theta+1440p\mid \theta\in D_r^{\text{day}},\ p\in P(T_{\max})\}.$$

$$A_{\text{transfer}}=
\left\{((h_r,m_r^-,t),(h_r,m_r^+,t+\tau^{\text{tr}}_r))\ \middle|\ 
r\in L^{\text{tr}},\ m_r^-,m_r^+\in M_{h_r},\ t\in D^{\text{tr}}_r,\ t+\tau^{\text{tr}}_r\leq T_{\max}\right\}.$$

### Kosten und Emissionen

Jede Kante $a$ erhält einen Kostenwert $c(a)$ und einen Emissionswert $e(a)$. Bei einer **Transportkante** hängen beide Werte von drei Größen ab:

| Größe | Bedeutung |
|---|---|
| $d_l$ | Länge der gefahrenen Verbindung in km |
| $c_{m_l}$ | Transportkosten des verwendeten Modus je Tonnenkilometer; Einheit: EUR/(t km) |
| $e_{m_l}$ | Transportemissionen des verwendeten Modus je Tonnenkilometer; Einheit: kg CO$_2$/(t km) |
| $w$ | transportiertes Gewicht in Tonnen |

Berechnung:


| Größe | Kosten | Emissionen |
|---|---|---|
| Formel | $c(a)=d_l\cdot c_{m_l}\cdot w$ | $e(a)=d_l\cdot e_{m_l}\cdot w$ |




## Optimierungsmodell

### Parameter (Gewichte und Schranken)

* $w_c, w_t, w_e \in [0, 1]$: Gewichte für Kosten, Zeit und Emissionen ($\sum w = 1.0$)
* $C_{\max}, T_{\max}, E_{\max}$: Normalisierungsfaktoren (`MAX_EST_...`)
* $B$: Maximales Budget (`MAX_PRICE`)
* $h_{\text{start}}, h_{\text{end}}$: Start- und Ziel-Hub
* $t_{\text{start}}, t_{\text{deadline}}$: Startzeitpunkt und späteste Ankunftszeit in Minuten

### Entscheidungsvariable

Für jede Kante $a \in A$ (wobei $A$ die Gesamtmenge aller generierten Arcs ist) definieren wir eine binäre Variable:

$$x_a = \begin{cases} 1, & \text{wenn Kante } a \text{ genutzt wird} \\ 0, & \text{sonst} \end{cases}$$

Jede Kante $a$ besitzt die Eigenschaften $c_a$ (Kosten), $d_a$ (Dauer/Time) und $e_a$ (Emissionen).

---

### Zielfunktion

Das Modell minimiert die gewichtete, normalisierte Summe aus Transportkosten, Transportdauer und CO₂-Emissionen entlang des gewählten Pfades:

$$\min_{x} \sum_{a \in A} x_a \cdot \left( w_c \cdot \frac{c_a}{C_{\max}} + w_t \cdot \frac{d_a}{T_{\max}} + w_e \cdot \frac{e_a}{E_{\max}} \right)$$

---

### Restriktionen

#### Budgetbeschränkung

Die Summe der Kosten aller gewählten Kanten darf das Budget $B$ nicht überschreiten:

$$\sum_{a \in A} x_a \cdot c_a \leq B$$

#### Flusserhaltung

Seien $N$ die Zeitknoten und $A\subseteq N\times N$ die gerichteten Kanten $a=(i,j)$.

$$A^{\mathrm{in}}(n)=\{(i,n)\in A\mid i\in N\},\qquad
A^{\mathrm{out}}(n)=\{(n,j)\in A\mid j\in N\}.$$

$$N_{\mathrm{start}}=\{(h,m,t)\in N\mid h=h_{\mathrm{start}},\ t=t_{\mathrm{start}}\},$$

$$N_{\mathrm{end}}=\{(h,m,t)\in N\mid h=h_{\mathrm{end}},\ t\leq t_{\mathrm{deadline}}\}.$$

$$\sum_{a\in A^{\mathrm{in}}(n)}x_a-\sum_{a\in A^{\mathrm{out}}(n)}x_a=0
\qquad \forall n\in N\setminus(N_{\mathrm{start}}\cup N_{\mathrm{end}}).$$

#### Startbedingung

$$\sum_{n \in N_{\text{start}}}\ \sum_{a \in A^{\text{out}}(n)} x_a=1.$$

#### Zielbedingung

$$\sum_{n \in N_{\text{end}}}\ \sum_{a \in A^{\text{in}}(n)} x_a=1.$$

## Implementierung

### Datenstruktur und Generierung des zeitexpandierten Netzes

Wir erstellen drei Arten von Kanten:
1. **Transport-Arcs**: Bewegung zwischen Hubs.
2. **Waiting-Arcs**: Verweilen an einem Hub (Lagerung).
3. **Transfer-Arcs**: Wechsel zwischen Verkehrsträgern an einem Hub mit individuell vorgegebener Transferdauer.

Das Netzwerk wird ereignisbasiert in Minuten erzeugt: Transport- und Transfer-Arcs erzeugen Abfahrts- und Ankunftsereignisse, und Wartekanten verbinden nur direkt aufeinanderfolgende Ereignisse desselben Hub-Modus-Zustands.

### Anforderung an das Eingabedatenformat

Das Modell erwartet die Daten in den vier Blöcken `hubs`, `mode_factors`, `arc_templates` und `shipments`. Alle Zeitangaben werden in Minuten gemessen. Tagesbezogene Abfahrtszeiten liegen im Intervall `0 <= departure_min < 1440` und werden für jeden Planungstag wiederholt.

#### Hubs

`hubs` ist eine Liste von Hub-Objekten mit `id`, `name` und `supported_modes`. Die `id` muss eindeutig sein; `supported_modes` enthält die am Hub verfügbaren Modi.

#### Modusfaktoren

`mode_factors` enthält je Modus `cost_per_ton_km` und `emissions_kg_per_ton_km`. Diese Faktoren werden für Transportkosten und Transportemissionen verwendet.

#### Arc-Templates

Jedes Element in `arc_templates` benötigt `id`, `arc_type`, `from`, `to`, `departure_minutes` und `duration_min`. Für `arc_type == "transport"` kommen `mode` und `dist` hinzu. Für `arc_type == "transfer"` kommen `from_mode` und `to_mode` hinzu; außerdem muss `from == to` gelten.

Wartekanten werden nicht als Input angegeben. Sie werden automatisch zwischen aufeinanderfolgenden Ereigniszeiten desselben Hub-Modus-Zustands erzeugt.

#### Sendungen

`shipments` enthält je Sendung `start_hub`, `end_hub`, `start_time`, `deadline`, `max_price`, `max_emissions` und `weight`. `start_time` und `deadline` sind Minutenwerte.

#### Datenquelle umschalten

Die Beispielwerte bleiben im Notebook erhalten. Über `DATA_SOURCE = "inline"` oder `DATA_SOURCE = "dataset"` kann zwischen den Notebook-Beispieldaten und `dataset/multimodal_network.json` gewechselt werden. Bei `DATA_SOURCE = "dataset"` werden `hubs`, `mode_factors`, `arc_templates` und `shipments` aus der JSON-Datei geladen; die Einzelrouten-Parameter werden aus der ersten Sendung abgeleitet.

In [6]:
import json
from pathlib import Path

try:
    import pulp
except ModuleNotFoundError:
    pulp = None

import numpy as np

# 1. Basisdaten
hubs = [
    {
        "id": "BER",
        "name": "Berlin",
        "supported_modes": {"road", "rail", "air", "ship"},
    },
    {
        "id": "HAM",
        "name": "Hamburg",
        "supported_modes": {"road", "rail", "air", "ship"},
    },
    {
        "id": "FRA",
        "name": "Frankfurt",
        "supported_modes": {"road", "rail", "air", "ship"},
    },
    {
        "id": "MUC",
        "name": "Munich",
        "supported_modes": {"road", "rail", "air"},
    },
]

hubs_by_id = {hub["id"]: hub for hub in hubs}
assert len(hubs_by_id) == len(hubs), "Hub-IDs have to be unique."

mode_factors = {
    "road": {"cost_per_ton_km": 1.20, "emissions_kg_per_ton_km": 0.09},
    "rail": {
        "cost_per_ton_km": 0.70,
        "emissions_kg_per_ton_km": 0.025,
    },
    "air": {"cost_per_ton_km": 3.50, "emissions_kg_per_ton_km": 0.60},
    "ship": {
        "cost_per_ton_km": 0.40,
        "emissions_kg_per_ton_km": 0.015,
    },
}


def hour(value):
    return int(value * 60)


def hours(values):
    return [hour(value) for value in values]


START_HUB = "BER"
END_HUB = "MUC"
START_TIME_MIN = hour(4)
DEADLINE_MIN = hour(36)
MAX_PRICE = 3000.0
MAX_EMISSIONS = None

# Jede Transport- und Transferkante besitzt ihren eigenen täglichen Fahrplan.
# departure_minutes=hours([6, 18]) bedeutet: Diese konkrete Kante ist täglich um 06:00 und 18:00 Uhr verfügbar.
arc_templates = [
    # BER -> HAM
    {
        "id": "BER_HAM_road",
        "arc_type": "transport",
        "from": "BER",
        "to": "HAM",
        "mode": "road",
        "dist": 290,
        "duration_min": hour(5),
        "departure_minutes": hours(range(0, 24)),
    },
    {
        "id": "BER_HAM_rail",
        "arc_type": "transport",
        "from": "BER",
        "to": "HAM",
        "mode": "rail",
        "dist": 285,
        "duration_min": hour(5),
        "departure_minutes": hours([6, 18]),
    },
    {
        "id": "BER_HAM_ship",
        "arc_type": "transport",
        "from": "BER",
        "to": "HAM",
        "mode": "ship",
        "dist": 350,
        "duration_min": hour(14),
        "departure_minutes": hours([4, 16]),
    },
    # HAM -> FRA
    {
        "id": "HAM_FRA_road",
        "arc_type": "transport",
        "from": "HAM",
        "to": "FRA",
        "mode": "road",
        "dist": 500,
        "duration_min": hour(8),
        "departure_minutes": hours(range(0, 24)),
    },
    {
        "id": "HAM_FRA_rail",
        "arc_type": "transport",
        "from": "HAM",
        "to": "FRA",
        "mode": "rail",
        "dist": 510,
        "duration_min": hour(9),
        "departure_minutes": hours([8, 20]),
    },
    {
        "id": "HAM_FRA_ship",
        "arc_type": "transport",
        "from": "HAM",
        "to": "FRA",
        "mode": "ship",
        "dist": 620,
        "duration_min": hour(25),
        "departure_minutes": hours([8, 20]),
    },
    # FRA -> MUC
    {
        "id": "FRA_MUC_road",
        "arc_type": "transport",
        "from": "FRA",
        "to": "MUC",
        "mode": "road",
        "dist": 400,
        "duration_min": hour(6),
        "departure_minutes": hours(range(0, 24)),
    },
    {
        "id": "FRA_MUC_rail",
        "arc_type": "transport",
        "from": "FRA",
        "to": "MUC",
        "mode": "rail",
        "dist": 410,
        "duration_min": hour(7),
        "departure_minutes": hours([7, 19]),
    },
    {
        "id": "FRA_MUC_air",
        "arc_type": "transport",
        "from": "FRA",
        "to": "MUC",
        "mode": "air",
        "dist": 380,
        "duration_min": hour(1),
        "departure_minutes": hours([8, 16]),
    },
    # BER -> FRA
    {
        "id": "BER_FRA_road",
        "arc_type": "transport",
        "from": "BER",
        "to": "FRA",
        "mode": "road",
        "dist": 550,
        "duration_min": hour(8),
        "departure_minutes": hours(range(0, 24)),
    },
    {
        "id": "BER_FRA_rail",
        "arc_type": "transport",
        "from": "BER",
        "to": "FRA",
        "mode": "rail",
        "dist": 540,
        "duration_min": hour(9),
        "departure_minutes": hours([6, 18]),
    },
    {
        "id": "BER_FRA_air",
        "arc_type": "transport",
        "from": "BER",
        "to": "FRA",
        "mode": "air",
        "dist": 500,
        "duration_min": hour(1),
        "departure_minutes": hours([9, 17]),
    },
    # BER -> MUC
    {
        "id": "BER_MUC_road",
        "arc_type": "transport",
        "from": "BER",
        "to": "MUC",
        "mode": "road",
        "dist": 600,
        "duration_min": hour(9),
        "departure_minutes": hours(range(0, 24)),
    },
    {
        "id": "BER_MUC_rail",
        "arc_type": "transport",
        "from": "BER",
        "to": "MUC",
        "mode": "rail",
        "dist": 585,
        "duration_min": hour(10),
        "departure_minutes": hours([6, 18]),
    },
    {
        "id": "BER_MUC_air",
        "arc_type": "transport",
        "from": "BER",
        "to": "MUC",
        "mode": "air",
        "dist": 520,
        "duration_min": hour(2),
        "departure_minutes": hours([9, 17]),
    },
    # HAM -> MUC
    {
        "id": "HAM_MUC_road",
        "arc_type": "transport",
        "from": "HAM",
        "to": "MUC",
        "mode": "road",
        "dist": 800,
        "duration_min": hour(12),
        "departure_minutes": hours(range(0, 24)),
    },
    {
        "id": "HAM_MUC_rail",
        "arc_type": "transport",
        "from": "HAM",
        "to": "MUC",
        "mode": "rail",
        "dist": 780,
        "duration_min": hour(13),
        "departure_minutes": hours([8, 20]),
    },
    {
        "id": "HAM_MUC_air",
        "arc_type": "transport",
        "from": "HAM",
        "to": "MUC",
        "mode": "air",
        "dist": 650,
        "duration_min": hour(2),
        "departure_minutes": hours([11, 19]),
    },
    # Transfer arcs: same physical hub, different mode state
    {
        "id": "BER_BER_road_rail_transfer",
        "arc_type": "transfer",
        "from": "BER",
        "to": "BER",
        "from_mode": "road",
        "to_mode": "rail",
        "departure_minutes": hours([4, 5, 6, 16, 17]),
        "duration_min": hour(2),
    },
    {
        "id": "BER_BER_rail_road_transfer",
        "arc_type": "transfer",
        "from": "BER",
        "to": "BER",
        "from_mode": "rail",
        "to_mode": "road",
        "departure_minutes": hours([9, 10, 21, 22]),
        "duration_min": hour(2),
    },
    {
        "id": "BER_BER_road_air_transfer",
        "arc_type": "transfer",
        "from": "BER",
        "to": "BER",
        "from_mode": "road",
        "to_mode": "air",
        "departure_minutes": hours([7, 8, 15, 16]),
        "duration_min": hour(1),
    },
    {
        "id": "BER_BER_air_road_transfer",
        "arc_type": "transfer",
        "from": "BER",
        "to": "BER",
        "from_mode": "air",
        "to_mode": "road",
        "departure_minutes": hours([11, 12, 19, 20]),
        "duration_min": hour(1),
    },
    {
        "id": "BER_BER_road_ship_transfer",
        "arc_type": "transfer",
        "from": "BER",
        "to": "BER",
        "from_mode": "road",
        "to_mode": "ship",
        "departure_minutes": hours([2, 3, 14, 15]),
        "duration_min": hour(3),
    },
    {
        "id": "BER_BER_ship_road_transfer",
        "arc_type": "transfer",
        "from": "BER",
        "to": "BER",
        "from_mode": "ship",
        "to_mode": "road",
        "departure_minutes": hours([8, 9, 20, 21]),
        "duration_min": hour(3),
    },
    {
        "id": "BER_BER_rail_ship_transfer",
        "arc_type": "transfer",
        "from": "BER",
        "to": "BER",
        "from_mode": "rail",
        "to_mode": "ship",
        "departure_minutes": hours([5, 17]),
        "duration_min": hour(3),
    },
    {
        "id": "BER_BER_ship_rail_transfer",
        "arc_type": "transfer",
        "from": "BER",
        "to": "BER",
        "from_mode": "ship",
        "to_mode": "rail",
        "departure_minutes": hours([11, 23]),
        "duration_min": hour(3),
    },
    {
        "id": "HAM_HAM_road_rail_transfer",
        "arc_type": "transfer",
        "from": "HAM",
        "to": "HAM",
        "from_mode": "road",
        "to_mode": "rail",
        "departure_minutes": hours([6, 7, 18, 19]),
        "duration_min": hour(2),
    },
    {
        "id": "HAM_HAM_rail_road_transfer",
        "arc_type": "transfer",
        "from": "HAM",
        "to": "HAM",
        "from_mode": "rail",
        "to_mode": "road",
        "departure_minutes": hours([11, 12, 23]),
        "duration_min": hour(2),
    },
    {
        "id": "HAM_HAM_road_air_transfer",
        "arc_type": "transfer",
        "from": "HAM",
        "to": "HAM",
        "from_mode": "road",
        "to_mode": "air",
        "departure_minutes": hours([9, 10, 17, 18]),
        "duration_min": hour(1),
    },
    {
        "id": "HAM_HAM_air_road_transfer",
        "arc_type": "transfer",
        "from": "HAM",
        "to": "HAM",
        "from_mode": "air",
        "to_mode": "road",
        "departure_minutes": hours([13, 14, 21, 22]),
        "duration_min": hour(1),
    },
    {
        "id": "HAM_HAM_road_ship_transfer",
        "arc_type": "transfer",
        "from": "HAM",
        "to": "HAM",
        "from_mode": "road",
        "to_mode": "ship",
        "departure_minutes": hours([6, 7, 18, 19]),
        "duration_min": hour(3),
    },
    {
        "id": "HAM_HAM_ship_road_transfer",
        "arc_type": "transfer",
        "from": "HAM",
        "to": "HAM",
        "from_mode": "ship",
        "to_mode": "road",
        "departure_minutes": hours([10, 11, 22, 23]),
        "duration_min": hour(3),
    },
    {
        "id": "HAM_HAM_rail_ship_transfer",
        "arc_type": "transfer",
        "from": "HAM",
        "to": "HAM",
        "from_mode": "rail",
        "to_mode": "ship",
        "departure_minutes": hours([7, 19]),
        "duration_min": hour(3),
    },
    {
        "id": "HAM_HAM_ship_rail_transfer",
        "arc_type": "transfer",
        "from": "HAM",
        "to": "HAM",
        "from_mode": "ship",
        "to_mode": "rail",
        "departure_minutes": hours([9, 21]),
        "duration_min": hour(3),
    },
    {
        "id": "FRA_FRA_road_rail_transfer",
        "arc_type": "transfer",
        "from": "FRA",
        "to": "FRA",
        "from_mode": "road",
        "to_mode": "rail",
        "departure_minutes": hours([5, 6, 17, 18]),
        "duration_min": hour(2),
    },
    {
        "id": "FRA_FRA_rail_road_transfer",
        "arc_type": "transfer",
        "from": "FRA",
        "to": "FRA",
        "from_mode": "rail",
        "to_mode": "road",
        "departure_minutes": hours([9, 10, 21]),
        "duration_min": hour(2),
    },
    {
        "id": "FRA_FRA_road_air_transfer",
        "arc_type": "transfer",
        "from": "FRA",
        "to": "FRA",
        "from_mode": "road",
        "to_mode": "air",
        "departure_minutes": hours([6, 7, 14, 15]),
        "duration_min": hour(1),
    },
    {
        "id": "FRA_FRA_air_road_transfer",
        "arc_type": "transfer",
        "from": "FRA",
        "to": "FRA",
        "from_mode": "air",
        "to_mode": "road",
        "departure_minutes": hours([10, 11, 18, 19]),
        "duration_min": hour(1),
    },
    {
        "id": "FRA_FRA_road_ship_transfer",
        "arc_type": "transfer",
        "from": "FRA",
        "to": "FRA",
        "from_mode": "road",
        "to_mode": "ship",
        "departure_minutes": hours([4, 5, 16, 17]),
        "duration_min": hour(3),
    },
    {
        "id": "FRA_FRA_ship_road_transfer",
        "arc_type": "transfer",
        "from": "FRA",
        "to": "FRA",
        "from_mode": "ship",
        "to_mode": "road",
        "departure_minutes": hours([10, 11, 22]),
        "duration_min": hour(3),
    },
    {
        "id": "FRA_FRA_rail_ship_transfer",
        "arc_type": "transfer",
        "from": "FRA",
        "to": "FRA",
        "from_mode": "rail",
        "to_mode": "ship",
        "departure_minutes": hours([6, 18]),
        "duration_min": hour(3),
    },
    {
        "id": "FRA_FRA_ship_rail_transfer",
        "arc_type": "transfer",
        "from": "FRA",
        "to": "FRA",
        "from_mode": "ship",
        "to_mode": "rail",
        "departure_minutes": hours([8, 20]),
        "duration_min": hour(3),
    },
    {
        "id": "MUC_MUC_road_rail_transfer",
        "arc_type": "transfer",
        "from": "MUC",
        "to": "MUC",
        "from_mode": "road",
        "to_mode": "rail",
        "departure_minutes": hours([4, 5, 19, 20]),
        "duration_min": hour(2),
    },
    {
        "id": "MUC_MUC_rail_road_transfer",
        "arc_type": "transfer",
        "from": "MUC",
        "to": "MUC",
        "from_mode": "rail",
        "to_mode": "road",
        "departure_minutes": hours([9, 10, 22]),
        "duration_min": hour(2),
    },
    {
        "id": "MUC_MUC_road_air_transfer",
        "arc_type": "transfer",
        "from": "MUC",
        "to": "MUC",
        "from_mode": "road",
        "to_mode": "air",
        "departure_minutes": hours([8, 9, 18, 19]),
        "duration_min": hour(1),
    },
    {
        "id": "MUC_MUC_air_road_transfer",
        "arc_type": "transfer",
        "from": "MUC",
        "to": "MUC",
        "from_mode": "air",
        "to_mode": "road",
        "departure_minutes": hours([12, 13, 22]),
        "duration_min": hour(1),
    },
]

# Datenquelle umschalten: "inline" nutzt die Beispielwerte oben, "dataset" lädt nur Netzwerkdaten.
# Sendungen werden später explizit an TimeExpandedFreightRoutingModel.solve(...) übergeben.
DATA_SOURCE = "inline"
DATASET_PATH = Path("dataset/multimodal_network.json")

inline_hubs = hubs
inline_mode_factors = mode_factors
inline_arc_templates = arc_templates


def resolve_dataset_path(path):
    candidates = [path, Path("..") / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Dataset file not found: {path}")


def normalize_hubs(raw_hubs):
    return [{**hub, "supported_modes": set(hub["supported_modes"])} for hub in raw_hubs]


if DATA_SOURCE == "dataset":
    dataset_path = resolve_dataset_path(DATASET_PATH)
    with dataset_path.open(encoding="utf-8") as file:
        dataset = json.load(file)

    hubs = normalize_hubs(dataset["hubs"])
    mode_factors = dataset["mode_factors"]
    arc_templates = dataset["arc_templates"]
elif DATA_SOURCE != "inline":
    raise ValueError(f"Unknown DATA_SOURCE: {DATA_SOURCE}")

hubs_by_id = {hub["id"]: hub for hub in hubs}
assert len(hubs_by_id) == len(hubs), "Hub-IDs have to be unique."

In [7]:
time_arcs = []
shipment_weight = 1.0  # in Tonnen
max_time_min = hour(100)  # Planungshorizont
effective_deadline_min = min(DEADLINE_MIN, max_time_min)
minutes_per_day = hour(24)
planning_days = range((max_time_min + minutes_per_day - 1) // minutes_per_day)
waiting_cost_per_hour = 5.0


def make_node(hub, mode, time_min):
    return f"{hub}_{mode}_{time_min}"


def node_parts(node):
    hub, mode, time_min = node.rsplit("_", 2)
    return hub, mode, int(time_min)


def format_time(time_min):
    day, minute_of_day = divmod(time_min, minutes_per_day)
    hour_of_day, minute = divmod(minute_of_day, 60)
    return f"Tag {day}, {hour_of_day:02d}:{minute:02d}"


event_times = {
    (hub["id"], mode): {0, max_time_min}
    for hub in hubs
    for mode in hub["supported_modes"]
}


def add_event(hub, mode, time_min):
    if 0 <= time_min <= max_time_min:
        event_times.setdefault((hub, mode), set()).add(time_min)


# Start- und Deadlinezeiten werden als Events ergänzt, damit das Modell dort exakt starten bzw. ankommen kann.
for mode in hubs_by_id[START_HUB]["supported_modes"]:
    add_event(START_HUB, mode, START_TIME_MIN)

for mode in hubs_by_id[END_HUB]["supported_modes"]:
    add_event(END_HUB, mode, effective_deadline_min)

# Datenkonsistenz: Arc-Templates dürfen nur vorhandene Hubs und Hub-Modi referenzieren.
for template in arc_templates:
    assert "arc_type" in template, f"{template['id']}: arc_type fehlt."
    arc_type = template["arc_type"]
    from_hub = hubs_by_id[template["from"]]
    to_hub = hubs_by_id[template["to"]]
    assert all(
        isinstance(dep_min, int) and 0 <= dep_min < minutes_per_day
        for dep_min in template["departure_minutes"]
    ), f"{template['id']}: enthält ungültige Abfahrtsminuten."
    assert (
        isinstance(template["duration_min"], int) and template["duration_min"] > 0
    ), f"{template['id']}: benötigt eine positive ganze Dauer in Minuten."

    if arc_type == "transport":
        mode = template["mode"]
        assert (
            template["dist"] >= 0
        ), f"{template['id']}: Distanz darf nicht negativ sein."
        assert (
            mode in from_hub["supported_modes"]
        ), f"{template['id']}: {mode} ist am Start-Hub nicht verfügbar."
        assert (
            mode in to_hub["supported_modes"]
        ), f"{template['id']}: {mode} ist am Ziel-Hub nicht verfügbar."
    elif arc_type == "transfer":
        assert (
            template["from"] == template["to"]
        ), f"{template['id']}: Transfer-Arcs müssen am selben Hub starten und enden."
        assert {template["from_mode"], template["to_mode"]}.issubset(
            from_hub["supported_modes"]
        ), f"{template['id']}: Transfer verwendet einen am Hub nicht verfügbaren Modus."
    else:
        raise ValueError(f"{template['id']}: unbekannter Arc-Typ {arc_type}.")

# 2.1 Transport Arcs: erzeugen Events an Abfahrt und Ankunft.
for template in arc_templates:
    if template["arc_type"] != "transport":
        continue

    from_hub = template["from"]
    to_hub = template["to"]
    mode = template["mode"]
    dist = template["dist"]
    duration = template["duration_min"]

    for day in planning_days:
        for dep_min in template["departure_minutes"]:
            start_min = day * minutes_per_day + dep_min
            arrival_min = start_min + duration
            if arrival_min <= max_time_min:
                add_event(from_hub, mode, start_min)
                add_event(to_hub, mode, arrival_min)
                time_arcs.append(
                    {
                        "service_id": template["id"],
                        "from": make_node(from_hub, mode, start_min),
                        "to": make_node(to_hub, mode, arrival_min),
                        "arc_type": "transport",
                        "type": "transport",
                        "mode": mode,
                        "dist": dist,
                        "cost": dist
                        * mode_factors[mode]["cost_per_ton_km"]
                        * shipment_weight,
                        "emissions": dist
                        * mode_factors[mode]["emissions_kg_per_ton_km"]
                        * shipment_weight,
                        "duration_min": duration,
                        "duration": duration,
                        "departure_min": start_min,
                        "arrival_min": arrival_min,
                    }
                )

# 2.2 Transfer Arcs: erzeugen Events am Startmodus und am Zielmodus.
for template in arc_templates:
    if template["arc_type"] != "transfer":
        continue

    hub = template["from"]
    m1 = template["from_mode"]
    m2 = template["to_mode"]
    transfer_duration = template["duration_min"]
    for day in planning_days:
        for dep_min in template["departure_minutes"]:
            start_min = day * minutes_per_day + dep_min
            arrival_min = start_min + transfer_duration
            if arrival_min <= max_time_min:
                add_event(hub, m1, start_min)
                add_event(hub, m2, arrival_min)
                time_arcs.append(
                    {
                        "service_id": template["id"],
                        "from": make_node(hub, m1, start_min),
                        "to": make_node(hub, m2, arrival_min),
                        "arc_type": "transfer",
                        "type": "transfer",
                        "mode": f"{m1}->{m2}",
                        "cost": 50.0 * shipment_weight,  # Fixe Handlingkosten
                        "emissions": 5.0 * shipment_weight,  # Emissionen beim Umschlag
                        "duration_min": transfer_duration,
                        "duration": transfer_duration,
                        "departure_min": start_min,
                        "arrival_min": arrival_min,
                    }
                )

# 2.3 Waiting Arcs: nur zwischen direkt aufeinanderfolgenden Events desselben Hub-Modus-Zustands.
for (hub_id, mode), times in sorted(event_times.items()):
    sorted_times = sorted(times)
    for start_min, arrival_min in zip(sorted_times, sorted_times[1:]):
        duration = arrival_min - start_min
        if duration <= 0:
            continue
        time_arcs.append(
            {
                "from": make_node(hub_id, mode, start_min),
                "to": make_node(hub_id, mode, arrival_min),
                "arc_type": "waiting",
                "type": "waiting",
                "mode": mode,
                "cost": waiting_cost_per_hour * (duration / 60) * shipment_weight,
                "emissions": 0.0,
                "duration_min": duration,
                "duration": duration,
                "departure_min": start_min,
                "arrival_min": arrival_min,
            }
        )

assert all(
    "arc_type" in arc for arc in time_arcs
), "Jeder generierte Arc braucht einen arc_type."
assert all(
    "duration_min" in arc for arc in time_arcs
), "Jeder generierte Arc braucht duration_min."

if DEADLINE_MIN > max_time_min:
    print(
        f"Deadline {format_time(DEADLINE_MIN)} liegt hinter dem Planungshorizont; "
        f"für die Suche wird {format_time(effective_deadline_min)} verwendet."
    )

print(
    f"Anzahl generierter Event-Zeitpunkte: {sum(len(times) for times in event_times.values())}"
)
print(f"Anzahl generierter Arcs: {len(time_arcs)}")

Anzahl generierter Event-Zeitpunkte: 869
Anzahl generierter Arcs: 1887


## Ergebnis und Auswertung

In [8]:
if route_solution is None and prob is not None:
    status = prob.solve(pulp.PULP_CBC_CMD(msg=0, timeLimit=60))
    status_name = pulp.LpStatus[status]
    if status_name == "Optimal":
        route_solution = {
            "status": "Optimal",
            "solver": route_solver,
            "arc_indices": [i for i in arc_indices if pulp.value(arc_vars[i]) > 0.5],
            "objective": pulp.value(prob.objective),
        }
    else:
        route_solution = {
            "status": status_name,
            "solver": route_solver,
            "arc_indices": [],
        }

if route_solution["status"] == "Optimal":
    print(f"Optimale Route gefunden ({route_solution['solver']}).\n")
    chosen_arcs = [time_arcs[i] for i in route_solution["arc_indices"]]
    chosen_arcs.sort(key=lambda arc: node_parts(arc["from"])[2])

    total_cost = 0.0
    total_eco = 0.0
    last_time = START_TIME_MIN

    for arc in chosen_arcs:
        _, _, from_time = node_parts(arc["from"])
        _, _, to_time = node_parts(arc["to"])
        print(
            f"[{arc['from']}] -> [{arc['to']}] | "
            f"Zeit: {format_time(from_time)} -> {format_time(to_time)} | "
            f"Modus: {arc['mode']:10} | Typ: {arc['type']:10} | "
            f"Dauer: {arc['duration_min']:4d} min | Kosten: {arc['cost']:10.2f} | CO2: {arc['emissions']:10.2f}"
        )
        total_cost += arc["cost"]
        total_eco += arc["emissions"]
        last_time = to_time

    print("\n" + "=" * 30)
    print(f"Gesamtkosten:   {total_cost:10.2f} EUR / Limit: {MAX_PRICE:.2f} EUR")
    if MAX_EMISSIONS is None:
        print(f"Gesamt-CO2:     {total_eco:10.2f} kg")
    else:
        print(f"Gesamt-CO2:     {total_eco:10.2f} kg / Limit: {MAX_EMISSIONS:.2f} kg")
    total_duration = last_time - START_TIME_MIN
    print(f"Ankunft:        {format_time(last_time):>10}")
    print(f"Dauer:          {total_duration:10d} min ({total_duration / 60:.1f} h)")
    print("=" * 30)
else:
    print(
        f"Keine Lösung gefunden. Status: {route_solution['status']} ({route_solution['solver']})"
    )

NameError: name 'route_solution' is not defined

## Modellgrenzen und weiterer Entwicklungsbedarf

### Vereinfachungen und Annahmen

- Es wird eine unteilbare Sendung betrachtet.
- Zeit wird in Minuten gemessen; Fahrzeiten stehen als `duration_min` direkt in den Arc-Daten
- Die Abfahrtszeiten in `departure_minutes` wiederholen sich an jedem Planungstag identisch
- Fahr-, Warte- und Transferzeiten sind deterministisch, Störungen werden nicht modelliert.

### Zu berücksichtigende Erweiterungen

- Mehrere Sendungen und Konsolidierung modellieren
- Wartekosten und Warteemissionen realitätsnäher abbilden, aktuell gelten $5\cdot w$ EUR je Stunde und $0$ kg CO$_2$
- Transferkosten und -emissionen nach Hub und Moduspaar differenzieren; die Transferdauer ist bereits je Transfer-Arc als `duration_min` modelliert.
- Transportkosten und -emissionen um Fixkosten erweitern
- Kapazitäten von Transportverbindungen, Hubs und Transfers ergänzen.
- Testdaten durch dokumentierte und validierte reale Daten ersetzen
- Die angenommenen Skalierungswerte `MAX_EST_COST`, `MAX_EST_TIME` und `MAX_EST_ECO` methodisch bestimmen

## Mehrere Sendungen

Für mehrere Sendungen wird das Modell als Mehrgüterflussproblem auf demselben zeitexpandierten Netzwerk formuliert. Jede Sendung darf einen eigenen Start-Hub, Ziel-Hub, späteste Ankunftszeit, Budgetwert und ein eigenes Gewicht besitzen.

### Zusätzliche Mengen und Parameter

| Symbol | Bedeutung |
|---|---|
| $K$ | Menge der Sendungen |
| $s_k$ | Start-Hub der Sendung $k\in K$ |
| $r_k$ | Ziel-Hub der Sendung $k\in K$ |
| $t_k^{\mathrm{start}}$ | frühester Startzeitpunkt der Sendung $k$ |
| $D_k$ | späteste zulässige Ankunftszeit der Sendung $k$ |
| $B_k$ | maximales Budget der Sendung $k$ |
| $w_k$ | Gewicht der Sendung $k$ in Tonnen |
| $U_a$ | Kapazität der Kante $a\in A$ in Tonnen |

Falls alle Sendungen denselben Startzeitpunkt verwenden, kann $t_k^{\mathrm{start}}=t_{\mathrm{start}}$ für alle $k\in K$ gesetzt werden. Die Start- und Ziel-Hubs bleiben trotzdem sendungsspezifisch.

Die Kosten und Emissionen einer Kante hängen nun von der jeweiligen Sendung ab, weil die Sendungen unterschiedliche Gewichte besitzen können:

$$c_{a,k}=c_a(w_k),\qquad e_{a,k}=e_a(w_k).$$

Für Transportkanten, die aus einer Verbindung $l$ entstehen, gilt beispielsweise:

$$c_{a,k}=d_l\cdot c_{m_l}\cdot w_k,$$

$$e_{a,k}=d_l\cdot e_{m_l}\cdot w_k.$$

### Entscheidungsvariable

Für jede Kante $a\in A$ und jede Sendung $k\in K$ wird eine binäre Variable definiert:

$$x_{a,k}=\begin{cases}1, & \text{wenn Sendung } k \text{ Kante } a \text{ nutzt} \\ 0, & \text{sonst.}\end{cases}$$

Damit kann jede Sendung eine eigene Route durch dasselbe Netzwerk wählen.

### Zielfunktion

Die Zielfunktion minimiert die Summe der gewichteten, normalisierten Kosten-, Zeit- und Emissionsanteile über alle Sendungen:

$$\min_x \sum_{k\in K}\sum_{a\in A} x_{a,k}\left(w_c\frac{c_{a,k}}{C_{\max}}+w_t\frac{\tau_a}{T_{\max}}+w_e\frac{e_{a,k}}{E_{\max}}\right).$$

Dabei ist $\tau_a$ die Dauer der Kante $a$.

### Restriktionen

#### Budgetbeschränkung

Jede Sendung muss ihr eigenes Budget einhalten:

$$\sum_{a\in A} x_{a,k}\cdot c_{a,k}\leq B_k\qquad \forall k\in K.$$

#### Flusserhaltung

Seien $N$ die Zeitknoten und $A\subseteq N\times N$ die gerichteten Kanten $a=(i,j)$.

$$A^{\mathrm{in}}(n)=\{(i,n)\in A\mid i\in N\},\qquad
A^{\mathrm{out}}(n)=\{(n,j)\in A\mid j\in N\}.$$

Für jede Sendung werden eigene Start- und Zielknoten definiert:

$$N_{k}^{\mathrm{start}}=\{(h,m,t)\in N\mid h=s_k,\ t=t_k^{\mathrm{start}}\},$$

$$N_{k}^{\mathrm{end}}=\{(h,m,t)\in N\mid h=r_k\}.$$

Für alle Zwischenknoten gilt die Flusserhaltung sendungsweise:

$$\sum_{a\in A^{\mathrm{in}}(n)}x_{a,k}-\sum_{a\in A^{\mathrm{out}}(n)}x_{a,k}=0
\qquad \forall k\in K,\ \forall n\in N\setminus\left(N_k^{\mathrm{start}}\cup N_k^{\mathrm{end}}\right).$$

#### Startbedingung

Jede Sendung muss genau einmal an ihrem eigenen Start-Hub starten:

$$\sum_{n\in N_k^{\mathrm{start}}}\sum_{a\in A^{\mathrm{out}}(n)}x_{a,k}=1
\qquad \forall k\in K.$$

#### Zielbedingung

Jede Sendung muss genau einmal an ihrem eigenen Ziel-Hub ankommen:

$$\sum_{n\in N_k^{\mathrm{end}}}\sum_{a\in A^{\mathrm{in}}(n)}x_{a,k}=1
\qquad \forall k\in K.$$

#### Deadlinebedingung

Schreibe $t(n)$ für die Zeitkomponente eines Knotens $n=(h,m,t)$. Da jede Sendung nach der Zielbedingung genau einmal an einem Zielknoten ankommt, kann die Deadline als eigene Restriktion formuliert werden:

$$\sum_{n\in N_k^{\mathrm{end}}}\sum_{a\in A^{\mathrm{in}}(n)} t(n)\,x_{a,k}\leq D_k
\qquad \forall k\in K.$$

#### Binärbedingungen

$$x_{a,k}\in\{0,1\}\qquad \forall a\in A,\ \forall k\in K.$$

#### Kapazitätsbeschränkung

Die gemeinsam genutzten Kanten des Netzwerks besitzen begrenzte Kapazitäten. Die Summe der Gewichte aller Sendungen, die dieselbe Kante nutzen, darf die Kapazität dieser Kante nicht überschreiten:

$$\sum_{k\in K} w_k x_{a,k}\leq U_a\qquad \forall a\in A.$$


In [ ]:
shipments = [
    {
        "start_hub": "BER",
        "end_hub": "MUC",
        "deadline": hour(36),
        "max_price": 3000,
        "max_emissions": 60,
        "weight": 1,
    },
    {
        "start_hub": "BER",
        "end_hub": "MUC",
        "deadline": hour(20),
        "max_price": 6000,
        "max_emissions": 90,
        "weight": 2,
    },
]

# Beispiel für eine zweite Sendung. Aktuell deaktiviert, weil im Netzwerk keine Rückrichtung MUC -> FRA modelliert ist.
# shipments.append({
#     "start_hub": "MUC",
#     "end_hub": "FRA",
#     "deadline": hour(25),
#     "max_price": 4000,
#     "max_emissions": 120,
#     "weight": 2,
# })

# Im späteren Klassenmodell werden Sendungen explizit an solve(...) übergeben.

### Konsolidierung mit Fixkosten, Fix-Emissionen und Fahrzeuganzahl

Für echte Konsolidierung wird zusätzlich zu den sendungsspezifischen Routing-Variablen $x_{a,k}$ eine Aktivierungsvariable $y_a$ eingeführt. Die Fixkosten und Fix-Emissionen einer Kante werden nur einmal gezahlt bzw. verursacht, wenn mindestens eine Sendung diese Kante nutzt. Mehrere Sendungen können sich dadurch eine aktivierte Verbindung teilen.

$$y_a=\begin{cases}1, & \text{wenn Kante }a\text{ aktiviert wird} \\ 0, & \text{sonst}\end{cases}$$

Für Road-Transportkanten wird zusätzlich eine ganzzahlige Variable $z_a$ eingeführt. Sie beschreibt die Anzahl eingesetzter LKWs auf der Kante:

$$z_a\in\{0,1,2,\ldots\}\qquad \forall a\in A^{\mathrm{road}}.$$

Mit Fixkosten $F_a$ und variablen Kosten $c_{a,k}^{\mathrm{var}}$ ergibt sich der kostenbezogene Anteil der Zielfunktion zu:

$$\sum_{a\in A\setminus A^{\mathrm{road}}}F_a y_a+\sum_{a\in A^{\mathrm{road}}}F_a z_a+\sum_{k\in K}\sum_{a\in A}c_{a,k}^{\mathrm{var}}x_{a,k}.$$

Analog ergeben sich mit Fix-Emissionen $G_a$ und variablen Emissionen $e_{a,k}^{\mathrm{var}}$ die Gesamtemissionen als:

$$\sum_{a\in A\setminus A^{\mathrm{road}}}G_a y_a+\sum_{a\in A^{\mathrm{road}}}G_a z_a+\sum_{k\in K}\sum_{a\in A}e_{a,k}^{\mathrm{var}}x_{a,k}.$$

Für jede Sendung kann zusätzlich ein individuelles CO2-Budget $E_k^{\max}$ modelliert werden. Es begrenzt die variablen Emissionen der von Sendung $k$ genutzten Route:

$$\sum_{a\in A}e_{a,k}^{\mathrm{var}}x_{a,k}\leq E_k^{\max}\qquad \forall k\in K.$$

Die Fix-Emissionen $G_a$ werden weiterhin in den Gesamtemissionen ausgewiesen. Da sie bei Konsolidierung gemeinsam verursacht werden, bräuchte ihre sendungsspezifische Zuordnung eine zusätzliche Allokationsregel.

Die Kapazitätsrestriktion koppelt Nutzung und Aktivierung. Für Nicht-Road-Kanten gilt:

$$\sum_{k\in K}w_k x_{a,k}\leq U_a y_a\qquad \forall a\in A\setminus A^{\mathrm{road}}.$$

Für Road-Transportkanten gilt dagegen mit LKW-Kapazität $Q_a$:

$$\sum_{k\in K}w_k x_{a,k}\leq Q_a z_a\qquad \forall a\in A^{\mathrm{road}}.$$


In [ ]:
CONSOLIDATION_ARC_LIMIT = 25_000

shipment_start_default = START_TIME_MIN if "START_TIME_MIN" in globals() else 0

shipments = [shipment for shipment in shipments if isinstance(shipment, dict)]

for idx, shipment in enumerate(shipments):
    shipment.setdefault("id", f"S{idx + 1}")
    shipment.setdefault("start_time", shipment_start_default)
    shipment.setdefault("max_emissions", 500.0)

# Fixkosten werden pro aktivierter zeitexpandierter Kante nur einmal gezahlt.
fixed_cost_by_type = {
    "transport": {"road": 150.0, "rail": 500.0, "air": 1200.0, "ship": 800.0},
    "waiting": 0.0,
    "transfer": 100.0,
}

# Fix-Emissionen werden pro aktivierter zeitexpandierter Kante nur einmal verursacht.
fixed_emissions_by_type = {
    "transport": {"road": 30.0, "rail": 80.0, "air": 250.0, "ship": 120.0},
    "waiting": 0.0,
    "transfer": 10.0,
}

# Kapazitäten in Tonnen. Für road bezeichnet der Wert die Kapazität eines LKWs.
capacity_by_type = {
    "transport": {"road": 10.0, "rail": 40.0, "air": 5.0, "ship": 80.0},
    "waiting": 100.0,
    "transfer": 25.0,
}


def get_fixed_cost(arc):
    if arc["type"] == "transport":
        return fixed_cost_by_type["transport"][arc["mode"]]
    return fixed_cost_by_type[arc["type"]]


def get_fixed_emissions(arc):
    if arc["type"] == "transport":
        return fixed_emissions_by_type["transport"][arc["mode"]]
    return fixed_emissions_by_type[arc["type"]]


def get_capacity(arc):
    if arc["type"] == "transport":
        return capacity_by_type["transport"][arc["mode"]]
    return capacity_by_type[arc["type"]]


def node_parts(node):
    hub, mode, t = node.rsplit("_", 2)
    return hub, mode, int(t)


arc_indices = list(range(len(time_arcs)))
shipment_indices = list(range(len(shipments)))
road_transport_arc_indices = [
    i
    for i in arc_indices
    if time_arcs[i]["type"] == "transport" and time_arcs[i]["mode"] == "road"
]
road_transport_arc_set = set(road_transport_arc_indices)
non_road_arc_indices = [i for i in arc_indices if i not in road_transport_arc_set]

total_shipment_weight = sum(shipment["weight"] for shipment in shipments)
max_road_vehicles = max(
    1,
    int(np.ceil(total_shipment_weight / capacity_by_type["transport"]["road"])),
)

nodes_in_network = set()
incoming_by_node = {}
outgoing_by_node = {}
for i, arc in enumerate(time_arcs):
    from_node = arc["from"]
    to_node = arc["to"]
    nodes_in_network.add(from_node)
    nodes_in_network.add(to_node)
    outgoing_by_node.setdefault(from_node, []).append(i)
    incoming_by_node.setdefault(to_node, []).append(i)

fixed_arc_cost = {i: get_fixed_cost(time_arcs[i]) for i in arc_indices}
fixed_arc_emissions = {i: get_fixed_emissions(time_arcs[i]) for i in arc_indices}
arc_capacity = {i: get_capacity(time_arcs[i]) for i in arc_indices}

# Die bestehenden time_arcs wurden mit shipment_weight=1.0 erzeugt.
# Deshalb dienen cost und emissions hier als Werte pro Tonne.
variable_cost = {
    (i, k): time_arcs[i]["cost"] * shipments[k]["weight"] / shipment_weight
    for i in arc_indices
    for k in shipment_indices
}
shipment_emissions = {
    (i, k): time_arcs[i]["emissions"] * shipments[k]["weight"] / shipment_weight
    for i in arc_indices
    for k in shipment_indices
}

prob_consolidation = pulp.LpProblem("Multimodal_Routing_Consolidation", pulp.LpMinimize)

x = pulp.LpVariable.dicts(
    "UseArc",
    [(i, k) for i in arc_indices for k in shipment_indices],
    cat=pulp.LpBinary,
)
y = pulp.LpVariable.dicts("ActivateArc", non_road_arc_indices, cat=pulp.LpBinary)
vehicle_count = pulp.LpVariable.dicts(
    "VehicleCount",
    road_transport_arc_indices,
    lowBound=0,
    upBound=max_road_vehicles,
    cat=pulp.LpInteger,
)

MAX_EST_FIXED_COST = 5000.0
MAX_EST_FIXED_ECO = 500.0
CONSOLIDATION_W_COST = 1.0
CONSOLIDATION_W_TIME = 0.0
CONSOLIDATION_W_ECO = 0.0

prob_consolidation += (
    pulp.lpSum(
        y[i]
        * (
            CONSOLIDATION_W_COST * (fixed_arc_cost[i] / MAX_EST_FIXED_COST)
            + CONSOLIDATION_W_ECO * (fixed_arc_emissions[i] / MAX_EST_FIXED_ECO)
        )
        for i in non_road_arc_indices
    )
    + pulp.lpSum(
        vehicle_count[i]
        * (
            CONSOLIDATION_W_COST * (fixed_arc_cost[i] / MAX_EST_FIXED_COST)
            + CONSOLIDATION_W_ECO * (fixed_arc_emissions[i] / MAX_EST_FIXED_ECO)
        )
        for i in road_transport_arc_indices
    )
    + pulp.lpSum(
        x[(i, k)]
        * (
            CONSOLIDATION_W_COST * (variable_cost[(i, k)] / MAX_EST_COST)
            + CONSOLIDATION_W_TIME * (time_arcs[i]["duration"] / MAX_EST_TIME)
            + CONSOLIDATION_W_ECO * (shipment_emissions[(i, k)] / MAX_EST_ECO)
        )
        for i in arc_indices
        for k in shipment_indices
    )
)

# Sendungsspezifische Budgets für variable, verursachungsgerechte Kosten.
for k, shipment in enumerate(shipments):
    prob_consolidation += (
        pulp.lpSum(variable_cost[(i, k)] * x[(i, k)] for i in arc_indices)
        <= shipment["max_price"]
    )

# Gemeinsames Gesamtbudget inklusive Fixkosten. Road-Fixkosten fallen pro LKW an.
total_budget = sum(shipment["max_price"] for shipment in shipments)
prob_consolidation += (
    pulp.lpSum(fixed_arc_cost[i] * y[i] for i in non_road_arc_indices)
    + pulp.lpSum(
        fixed_arc_cost[i] * vehicle_count[i] for i in road_transport_arc_indices
    )
    + pulp.lpSum(
        variable_cost[(i, k)] * x[(i, k)] for i in arc_indices for k in shipment_indices
    )
    <= total_budget
)

# Sendungsspezifisches CO2-Budget für die variablen Emissionen der Route.
for k, shipment in enumerate(shipments):
    prob_consolidation += (
        pulp.lpSum(shipment_emissions[(i, k)] * x[(i, k)] for i in arc_indices)
        <= shipment["max_emissions"]
    )

# Kapazität und Aktivierung: Nicht-Road-Kanten sind nur bei y_i=1 nutzbar.
for i in non_road_arc_indices:
    prob_consolidation += (
        pulp.lpSum(shipments[k]["weight"] * x[(i, k)] for k in shipment_indices)
        <= arc_capacity[i] * y[i]
    )
    prob_consolidation += y[i] <= pulp.lpSum(x[(i, k)] for k in shipment_indices)

# Road-Transportkanten können mehrere LKWs einsetzen.
for i in road_transport_arc_indices:
    prob_consolidation += (
        pulp.lpSum(shipments[k]["weight"] * x[(i, k)] for k in shipment_indices)
        <= arc_capacity[i] * vehicle_count[i]
    )
    prob_consolidation += vehicle_count[i] <= max_road_vehicles * pulp.lpSum(
        x[(i, k)] for k in shipment_indices
    )

for k, shipment in enumerate(shipments):
    start_nodes = {
        node
        for node in nodes_in_network
        if node_parts(node)[0] == shipment["start_hub"]
        and node_parts(node)[2] == shipment["start_time"]
    }
    end_nodes = {
        node for node in nodes_in_network if node_parts(node)[0] == shipment["end_hub"]
    }

    prob_consolidation += (
        pulp.lpSum(
            x[(i, k)] for node in start_nodes for i in outgoing_by_node.get(node, [])
        )
        == 1
    )

    prob_consolidation += (
        pulp.lpSum(
            x[(i, k)] for node in end_nodes for i in incoming_by_node.get(node, [])
        )
        == 1
    )

    prob_consolidation += (
        pulp.lpSum(
            node_parts(node)[2] * x[(i, k)]
            for node in end_nodes
            for i in incoming_by_node.get(node, [])
        )
        <= shipment["deadline"]
    )

    for node in nodes_in_network - start_nodes - end_nodes:
        prob_consolidation += pulp.lpSum(
            x[(i, k)] for i in incoming_by_node.get(node, [])
        ) == pulp.lpSum(x[(i, k)] for i in outgoing_by_node.get(node, []))

status = prob_consolidation.solve(pulp.PULP_CBC_CMD(msg=0))

if pulp.LpStatus[status] == "Optimal":
    print("Optimale konsolidierte Routen gefunden.\n")
    total_fixed_cost = sum(
        fixed_arc_cost[i] for i in non_road_arc_indices if pulp.value(y[i]) > 0.5
    ) + sum(
        fixed_arc_cost[i] * pulp.value(vehicle_count[i])
        for i in road_transport_arc_indices
    )
    total_fixed_emissions = sum(
        fixed_arc_emissions[i] for i in non_road_arc_indices if pulp.value(y[i]) > 0.5
    ) + sum(
        fixed_arc_emissions[i] * pulp.value(vehicle_count[i])
        for i in road_transport_arc_indices
    )
    total_variable_cost = sum(
        variable_cost[(i, k)] * pulp.value(x[(i, k)])
        for i in arc_indices
        for k in shipment_indices
    )
    total_variable_emissions = sum(
        shipment_emissions[(i, k)] * pulp.value(x[(i, k)])
        for i in arc_indices
        for k in shipment_indices
    )

    for k, shipment in enumerate(shipments):
        chosen_arc_indices = [i for i in arc_indices if pulp.value(x[(i, k)]) > 0.5]
        chosen_arcs = [time_arcs[i] for i in chosen_arc_indices]
        chosen_arcs.sort(key=lambda arc: int(arc["from"].split("_")[-1]))
        shipment_variable_emissions = sum(
            shipment_emissions[(i, k)] for i in chosen_arc_indices
        )

        print(
            f"Sendung {shipment['id']}: {shipment['start_hub']} -> {shipment['end_hub']}"
        )
        print(
            f"  CO2 der Route: {shipment_variable_emissions:.2f} kg / "
            f"Limit: {shipment['max_emissions']:.2f} kg"
        )
        for arc in chosen_arcs:
            _, _, from_time = node_parts(arc["from"])
            _, _, to_time = node_parts(arc["to"])
            print(
                f"  [{arc['from']}] -> [{arc['to']}] | "
                f"Zeit: {format_time(from_time)} -> {format_time(to_time)} | "
                f"Modus: {arc['mode']:10} | Typ: {arc['type']:10} | "
                f"Dauer: {arc['duration_min']:4d} min"
            )
        print()

    used_road_vehicles = [
        (i, int(round(pulp.value(vehicle_count[i]))))
        for i in road_transport_arc_indices
        if pulp.value(vehicle_count[i]) > 0.5
    ]
    if used_road_vehicles:
        print("Eingesetzte LKWs:")
        for i, count in used_road_vehicles:
            arc = time_arcs[i]
            print(f"  {count} x [{arc['from']}] -> [{arc['to']}]")
        print()

    print(f"Aktivierte Fixkosten: {total_fixed_cost:10.2f} EUR")
    print(f"Variable Kosten:      {total_variable_cost:10.2f} EUR")
    print(f"Gesamtkosten:         {total_fixed_cost + total_variable_cost:10.2f} EUR")
    print(f"Fix-Emissionen:       {total_fixed_emissions:10.2f} kg CO2")
    print(f"Variable Emissionen:  {total_variable_emissions:10.2f} kg CO2")
    print(
        f"Gesamtemissionen:     {total_fixed_emissions + total_variable_emissions:10.2f} kg CO2"
    )
else:
    print(f"Keine konsolidierte Lösung gefunden. Status: {pulp.LpStatus[status]}")

NameError: name 'shipments' is not defined

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from freight_routing.data_models import (
    ArcTemplate,
    ArcType,
    Hub,
    ModeFactor,
    NetworkData,
    Shipment,
    TransferArcTemplate,
    TransportArcTemplate,
    UserArcTemplate,
    _TimedArc,
)

In [2]:
from freight_routing.data_loader import NetworkDataLoader
from freight_routing.model import TimeExpandedFreightRoutingModel

network_data = NetworkDataLoader.from_json("../dataset/multimodal_network.json")

network_data.summary()

model = TimeExpandedFreightRoutingModel(network_data)
model.build(10)

model.summary()

Summary NetworkData:
hubs=870
arcs=36272
modes=4
Summary TimeExpandedFreightRoutingModel:
planning_days=10
planning_horizon_min=14400
nodes=977857
arcs=1766713
  - transport_arcs=685066
  - transfer_arcs=105920
  - waiting_arcs=975727


In [ ]:
from freight_routing.data_models import Shipment

test_shipments = [
    Shipment(
        id="shipment_1",
        start_hub="BER_3970",
        end_hub="MUE_3972",
        start_time=0,
        deadline=210060,
        max_price=300000.0,
        max_emissions=600000.0,
        weight=0.6,
    )
]

result = model.solve(test_shipments)

print(f"Solver Status: {result.status}")
print(f"Optimal Solution Found: {result.is_optimal}")
if result.is_optimal:
    print(
        f"Total Cost: {result.total_cost:.2f} EUR (Fixed: {result.total_fixed_cost:.2f}, Variable: {result.total_variable_cost:.2f})"
    )
    print(
        f"Total Emissions: {result.total_emissions:.2f} kg CO2 (Fixed: {result.total_fixed_emissions:.2f}, Variable: {result.total_variable_emissions:.2f})"
    )
    print(f"Total Transit Time: {result.total_time} min")

    print("\nRoutes per shipment:")
    for shipment_id, route in result.shipment_routes.items():
        print(f"\nShipment {shipment_id}:")
        for arc in route:
            print(
                f"  [{arc.from_node}] -> [{arc.to_node}] | Mode: {arc.mode:10} | Type: {arc.arc_type.value:8} | Cost: {arc.cost:6.2f} EUR"
            )